### Bayes Model Selection

The notebook implement related stuffs mentioned in https://www.overleaf.com/project/6a7592f575d67477ad61c188

For this notebook, the author will choose StatLo as the dataset for analyzing 

In [51]:
from libs import MCMC, AcceptanceTracker, SMC 

### Data Preprocessing

In [60]:
from ucimlrepo import fetch_ucirepo

statlog_heart = fetch_ucirepo(id=145)
X = statlog_heart.data.features
y = statlog_heart.data.targets 
X.describe()

,age,sex,chest-pain,rest-bp,serum-chol,fasting-blood-sugar,electrocardiographic,max-heart-rate,angina,oldpeak,slope,major-vessels,thal
count,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.00000,270.000000,270.000000,270.000000
mean,54.433333,0.677778,3.174074,131.344444,249.659259,0.148148,1.022222,149.677778,0.329630,1.05000,1.585185,0.670370,4.696296
std,9.109067,0.468195,0.950090,17.861608,51.686237,0.355906,0.997891,23.165717,0.470952,1.14521,0.614390,0.943896,1.940659
min,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.00000,1.000000,0.000000,3.000000
25%,48.000000,0.000000,3.000000,120.000000,213.000000,0.000000,0.000000,133.000000,0.000000,0.00000,1.000000,0.000000,3.000000
50%,55.000000,1.000000,3.000000,130.000000,245.000000,0.000000,2.000000,153.500000,0.000000,0.80000,2.000000,0.000000,3.000000
75%,61.000000,1.000000,4.000000,140.000000,280.000000,0.000000,2.000000,166.000000,1.000000,1.60000,2.000000,1.000000,7.000000
max,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.20000,3.000000,3.000000,7.000000


Some basic stats on the features 

In [61]:
X.head()

,age,sex,chest-pain,rest-bp,serum-chol,fasting-blood-sugar,electrocardiographic,max-heart-rate,angina,oldpeak,slope,major-vessels,thal
0,70.0,1.0,4.0,130.0,322.0,0.0,2.0,109.0,0.0,2.4,2.0,3.0,3.0
1,67.0,0.0,3.0,115.0,564.0,0.0,2.0,160.0,0.0,1.6,2.0,0.0,7.0
2,57.0,1.0,2.0,124.0,261.0,0.0,0.0,141.0,0.0,0.3,1.0,0.0,7.0
3,64.0,1.0,4.0,128.0,263.0,0.0,0.0,105.0,1.0,0.2,2.0,1.0,7.0
4,74.0,0.0,2.0,120.0,269.0,0.0,2.0,121.0,1.0,0.2,1.0,1.0,3.0


In [62]:
y.head()

,heart-disease
0,2
1,1
2,2
3,1
4,1


In [63]:
print(f"no columns: {len(X.columns)}")
print(X.columns.tolist())

no columns: 13
['age', 'sex', 'chest-pain', 'rest-bp', 'serum-chol', 'fasting-blood-sugar', 'electrocardiographic', 'max-heart-rate', 'angina', 'oldpeak', 'slope', 'major-vessels', 'thal']


Now, we will conduct data clean up with:

- Keep 8 columns for predictor
- Define binary target
- Standardize every non-intercept column

In [64]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
chosen_columns = [0, 1, 3, 4, 5, 7, 8, 9]
scaled_columns = [0, 3, 4, 7, 9]
X.iloc[:, scaled_columns] = scaler.fit_transform(
    X.iloc[:, scaled_columns]
)

chosen_X = X.iloc[: ,chosen_columns]
chosen_y = y.copy()
chosen_y["heart-disease"] = chosen_y["heart-disease"].replace(
    {
        2: 1,
        1: 0, 
    }
)

In [65]:
chosen_X.head()

,age,sex,rest-bp,serum-chol,fasting-blood-sugar,max-heart-rate,angina,oldpeak
0,1.712094,1.0,-0.075410,1.402212,0.0,-1.759208,0.0,1.181012
1,1.382140,0.0,-0.916759,6.093004,0.0,0.446409,0.0,0.481153
2,0.282294,1.0,-0.411950,0.219823,0.0,-0.375291,0.0,-0.656118
3,1.052186,1.0,-0.187590,0.258589,0.0,-1.932198,1.0,-0.743600
4,2.152032,0.0,-0.636310,0.374890,0.0,-1.240239,1.0,-0.743600


In [58]:
chosen_y

,heart-disease
0,1
1,0
2,1
3,0
4,0
...,...
265,0
266,0
267,0
268,0
